## CAM(Class Activation Maps)

- In XAI, they have come to refer a collection of methods that produce `heat maps` or `salency maps`. These show the most important pixels or regions in an image that the model has used to make the classification.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import cv2 as cv
import torch

import glob
from huggingface_hub import hf_hub_download

from modelling.datasets import ImageDataset
from modelling.network import CNNWithGAP

In [ ]:
# Download the model directly from hugging face hub
# model_path=hf_hub_download(
#     repo_id='a-data-odyssey/XAI-for-CV-models',
#     filename='models/pot_plant_classifier_gap/model.pth'
# )

Visit the link `https://huggingface.co/a-data-odyssey/XAI-for-CV-models/` and then accept and request the access for repo.

The `model.pth` will be downloaded automatically and import it into the current working folder.

In [ ]:
# Load the model
model=CNNWithGAP()
model_path='model.pth'
model.load_state_dict(torch.load(model_path))

In [ ]:
device=torch.device('mps' if torch.backends.mps.is_built()
                    else 'cuda' if torch.cuda.is_available()
                    else 'cpu')

model.to(device=device)

In [ ]:
model.eval()

In [ ]:
base_path='./data/pot_plants'

plant_names=['rudo', 'baya', 'greg', 'yuki']
num_classes=len(plant_names)

# Load the data
test_paths=glob.glob(base_path+"/train_valid/*.jpg")
test_data=ImageDataset(test_paths, class_names=plant_names)

In [ ]:
# Get random instance
image, target=test_data.__getitem__(0)

# Format input
input_img=image.unsqueeze(0).to(device)

# Format target
# argmax(): when the target is one-hot vector
target=torch.argmax(target).item()
target_name=plant_names[target]

# Get the prediction
output=model(input_img)
pred=torch.argmax(output).item()
pred_name=plant_names[pred]

# Display prediction
rgb_image=image.permute(1, 2, 0).numpy()
plt.imshow(rgb_image)
plt.title(f"Target: {target_name} ({target})\nPred: {pred_name} ({pred})")
plt.axis('off')

### Generating a CAM

In [ ]:
# Get the weights that connect GAP layer to output for class 1
gap_weights=model.fc[1].weight[1]
print(gap_weights.shape)

- Among 64, 1 value for each of the feature maps in that final convolutional layer.

In [ ]:
# Get final conv layer
final_conv_layer=model.conv_layers[-2]

# Hook to get the feature map from the last conv layer
feature_maps=[]

def hook_fn(module, input, output):
    feature_maps.append(output)

hook_handle=final_conv_layer.register_forward_hook(hook_fn)

# Forward pass to get feature maps
model(input_img)

# Remove the hook
hook_handle.remove()

print(feature_maps[0].shape)

In [ ]:
# Extract feature maps and GAP weights
feature_maps=feature_maps[0].squeeze(0).detach().cpu().numpy()
gap_weights=gap_weights.detach().cpu().numpy()

# Compute the CAM
cam=np.zeros(feature_maps.shape[1:], dtype=np.float32)
for i, w in enumerate(gap_weights):
    cam+=w*feature_maps[i]

In [ ]:
# Apply ReLU on cam for better visualization
cam=np.maximum(cam, 0)

# Normalize cam for visualization
cam=cam-np.min(cam)
cam=cam/np.max(cam)

In [ ]:
# Output class activation map
plt.imshow(cam, cmap='jet', alpha=0.5)
plt.title('CAM')
plt.axis('off')

In [ ]:
fig, ax=plt.subplots(nrows=1, ncols=2, figsize=(8, 5))

ax[0].imshow(rgb_image)
ax[0].set_title('Input Image')
ax[0].axis('off')

ax[1].imshow(cam, cmap='jet', alpha=0.5)
ax[1].set_title('CAM Image')
ax[1].axis('off')

plt.tight_layout()
plt.show()

### CAM Function

In [ ]:
def transform_image(img):
    transformed_img=img.permute(1, 2, 0).detach().cpu().numpy()
    transformed_img=(transformed_img-np.min(transformed_img))/(np.max(transformed_img)-np.min(transformed_img))
    return transformed_img

In [ ]:
model=CNNWithGAP()
model_path='model.pth'
model.load_state_dict(torch.load(model_path))

In [ ]:
def generate_cam(model, input_image, class_idx):
    final_conv_layer=model.conv_layers[-1]
    gap_weights=model.fc[1].weight[class_idx]

    feature_maps=[]
    def hook_fn(module, input, output):
        feature_maps.append(output)

    hook_handle=final_conv_layer.register_forward_hook(hook_fn)

    model(input_image)

    hook_handle.remove()

    # Extract feature maps and GAP weights
    feature_maps=feature_maps[0].squeeze(0).detach().cpu().numpy()
    gap_weights=gap_weights.detach().cpu().numpy()

    # Compute the CAM
    cam=np.zeros(feature_maps.shape[1:], dtype=np.float32)
    for i, w in enumerate(gap_weights):
        cam+=w*feature_maps[i]

    # Apply ReLU on cam for better visualization
    cam=np.maximum(cam, 0)

    # Normalize cam for visualization
    cam=cam-np.min(cam)
    cam=cam/np.max(cam)

    return cam

class_idx=2

test_data=ImageDataset(test_paths, class_names=plant_names)
image, target=test_data.__getitem__(class_idx)

input_image=image.unsqueeze(0).to(device)
cam=generate_cam(model, input_image, class_idx)

original_image=transform_image(image)

fig, ax=plt.subplots(nrows=1, ncols=2, figsize=(8, 4))

ax[0].imshow(original_image)
ax[0].set_title("Original Image")
ax[0].axis('off')

ax[1].imshow(original_image, alpha=0.5)
ax[1].imshow(cam, cmap='jet', alpha=0.5)
ax[1].set_title("CAM Image")
ax[1].axis('off')